In [3]:
pip install transformers datasets torch scikit-learn

  Using cached datasets-3.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 

In [4]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DistilBertTokenizer, DistilBertForSequenceClassification
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score
import numpy as np

In [5]:


# Load the cleaned IMDB CSV file
imdb_df = pd.read_csv('IMDB_Dataset_Cleaned_Featured.csv')
sst2_df = pd.read_csv('SST2_Dataset_Cleaned_Featured.csv')

# Convert pandas DataFrame to datasets Dataset
imdb = Dataset.from_pandas(imdb_df)
sst2 = Dataset.from_pandas(sst2_df)

In [6]:
# Initialize BERT tokenizer
'''
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Function to preprocess data
def preprocess_data(examples):
    # Tokenize the reviews
    tokenized = tokenizer(examples['review'], padding=True, truncation=True, max_length=512)

    # Additional features: Example with `num_nouns`, `num_verbs`, etc. (just placeholders)
    tokenized['num_nouns'] = examples['num_nouns']
    tokenized['num_verbs'] = examples['num_verbs']
    tokenized['sentiment_shift'] = examples['sentiment_shift']

    return tokenized

# Apply preprocessing to both training and testing datasets
imdb = imdb.map(preprocess_data, batched=True)
sst2 = sst2.map(preprocess_data, batched=True)

# Set format for PyTorch
imdb.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'num_nouns', 'num_verbs', 'sentiment_shift'])
sst2.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'num_nouns', 'num_verbs', 'sentiment_shift'])
'''

# Initialize DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Function to preprocess data
def preprocess_data(examples):
    # Tokenize the reviews
    tokenized = tokenizer(examples['review'], padding="max_length", truncation=True, max_length=512)

    # Additional features: Example with `num_nouns`, `num_verbs`, etc. (just placeholders)
    tokenized['num_nouns'] = examples['num_nouns']
    tokenized['num_verbs'] = examples['num_verbs']
    tokenized['sentiment_shift'] = examples['sentiment_shift']

    return tokenized

# Apply preprocessing to both training and testing datasets
imdb = imdb.map(preprocess_data, batched=True)
sst2 = sst2.map(preprocess_data, batched=True)

# Set format for PyTorch
imdb.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'num_nouns', 'num_verbs', 'sentiment_shift'])
sst2.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'num_nouns', 'num_verbs', 'sentiment_shift'])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/68221 [00:00<?, ? examples/s]

In [7]:
'''
class BERTWithExtraFeatures(torch.nn.Module):
    def __init__(self):
        super(BERTWithExtraFeatures, self).__init__()
        self.bert = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
        self.dropout = torch.nn.Dropout(0.1)
        self.fc = torch.nn.Linear(3, 2)  # Assuming we add 3 extra features (num_nouns, num_verbs, sentiment_shift)

    def forward(self, input_ids, attention_mask, num_nouns, num_verbs, sentiment_shift):
        # Pass through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        # Concatenate additional features to BERT's output
        extra_features = torch.stack([num_nouns, num_verbs, sentiment_shift], dim=-1)
        combined_input = torch.cat((pooled_output, extra_features), dim=-1)

        # Pass through a dropout layer and the final classification layer
        combined_input = self.dropout(combined_input)
        logits = self.fc(combined_input)

        return logits
'''

from transformers import BertForSequenceClassification, BertModel, DistilBertForSequenceClassification, DistilBertModel
import torch
import torch.nn as nn

'''
class BERTWithExtraFeatures(BertForSequenceClassification):
    def __init__(self, config, num_extra_features):
        super().__init__(config)
        self.num_extra_features = num_extra_features
        self.extra_features_layer = nn.Linear(num_extra_features, 1)  # Convert extra features to scalar

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None,
                num_nouns=None, num_verbs=None, sentiment_shift=None):
        # Get the outputs from the original BERT model
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask,
                                  token_type_ids=token_type_ids, labels=labels)

        # Process extra features if they exist
        if num_nouns is not None and num_verbs is not None and sentiment_shift is not None:
            # Concatenate extra features into a single tensor
            extra_features = torch.cat((num_nouns.unsqueeze(1), num_verbs.unsqueeze(1), sentiment_shift.unsqueeze(1)), dim=1)

            # Pass extra features through the extra_features_layer
            extra_features_output = self.extra_features_layer(extra_features)  # Shape: (batch_size, 1)

            # Expand extra features to match logits shape (batch_size, num_classes)
            extra_features_output = extra_features_output.expand(-1, outputs.logits.size(1))  # Shape: (batch_size, num_classes)

            # Add the extra features to the logits
            outputs.logits += extra_features_output

        return outputs



# Define the number of extra features (e.g., num_nouns, num_verbs, sentiment_shift)
num_extra_features = 3  # Adjust this based on the number of additional features you have

# Instantiate the model with the BERT configuration and the extra features count
model = BERTWithExtraFeatures.from_pretrained('bert-base-uncased', num_extra_features=num_extra_features)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Move model to GPU
model.to(device)
'''
'''
class DistilBERTWithExtraFeatures(DistilBertForSequenceClassification):
    def __init__(self, config, num_extra_features):
        super().__init__(config)
        self.num_extra_features = num_extra_features
        self.extra_features_layer = nn.Linear(num_extra_features, 1)  # Convert extra features to scalar

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None,
                num_nouns=None, num_verbs=None, sentiment_shift=None):
        # Get the outputs from the original DistilBERT model
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask,
                                  token_type_ids=token_type_ids, labels=labels)

        # Process extra features if they exist
        if num_nouns is not None and num_verbs is not None and sentiment_shift is not None:
            # Concatenate extra features into a single tensor
            extra_features = torch.cat((num_nouns.unsqueeze(1), num_verbs.unsqueeze(1), sentiment_shift.unsqueeze(1)), dim=1)

            # Pass extra features through the extra_features_layer
            extra_features_output = self.extra_features_layer(extra_features)  # Shape: (batch_size, 1)

            # Expand extra features to match logits shape (batch_size, num_classes)
            extra_features_output = extra_features_output.expand(-1, outputs.logits.size(1))  # Shape: (batch_size, num_classes)

            # Add the extra features to the logits
            outputs.logits += extra_features_output

        return outputs
'''

class DistilBERTWithExtraFeatures(DistilBertForSequenceClassification):
    def __init__(self, config, num_extra_features):
        super().__init__(config)
        self.num_extra_features = num_extra_features
        self.extra_features_layer = nn.Linear(num_extra_features, 1)  # Convert extra features to scalar

    def forward(self, input_ids=None, attention_mask=None, labels=None,
                num_nouns=None, num_verbs=None, sentiment_shift=None):
        # Get the outputs from the original DistilBERT model
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        # Process extra features if they exist
        if num_nouns is not None and num_verbs is not None and sentiment_shift is not None:
            # Concatenate extra features into a single tensor
            extra_features = torch.cat((num_nouns.unsqueeze(1), num_verbs.unsqueeze(1), sentiment_shift.unsqueeze(1)), dim=1)

            # Pass extra features through the extra_features_layer
            extra_features_output = self.extra_features_layer(extra_features)  # Shape: (batch_size, 1)

            # Expand extra features to match logits shape (batch_size, num_classes)
            extra_features_output = extra_features_output.expand(-1, outputs.logits.size(1))  # Shape: (batch_size, num_classes)

            # Add the extra features to the logits
            outputs.logits += extra_features_output

        return outputs

# Define the number of extra features (e.g., num_nouns, num_verbs, sentiment_shift)
num_extra_features = 3  # Adjust this based on the number of additional features you have

# Instantiate the model with the DistilBERT configuration and the extra features count
model = DistilBERTWithExtraFeatures.from_pretrained('distilbert-base-uncased', num_extra_features=num_extra_features)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Move model to GPU
model.to(device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBERTWithExtraFeatures were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'extra_features_layer.bias', 'extra_features_layer.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


DistilBERTWithExtraFeatures(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

In [ ]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="epoch",     # Evaluate after each epoch
    save_strategy="epoch",           # Save model after each epoch
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',      # <-- Required for early stopping
    greater_is_better=True                 # <-- True if higher = better (e.g., accuracy)
)

trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)], # stop if no visible improvements after 2 cons. runs
    train_dataset=imdb,         # training dataset
    eval_dataset=sst2,     # evaluation dataset
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}
)

trainer.train()

results = trainer.evaluate(sst2)
print("Test accuracy:", results['eval_accuracy'])

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
